1，从env文件中读取配置的大模型相关信息，API密钥，模型名称，API接口地址
2，检查这些重要信息有没有漏掉。
3，创建一个ChatOpenAI实例，用于调用大模型。

In [1]:
import json
import os

from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(find_dotenv())

missing = [name for name in ("LLM_BASE_URL", "LLM_API_KEY", "LLM_MODEL") if not os.environ.get(name)]
if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Copy .env.example to .env in the repository root and fill them in."
    )

llm = ChatOpenAI(
    model=os.environ["LLM_MODEL"],
    base_url=os.environ["LLM_BASE_URL"],
    api_key=os.environ["LLM_API_KEY"],
    extra_body=json.loads(os.environ.get("LLM_EXTRA_BODY") or "null"),
)
print(f"Model client ready: {os.environ['LLM_MODEL']} @ {os.environ['LLM_BASE_URL']}")

Model client ready: deepseek-v4-pro @ https://api.deepseek.com/v1


定义了三个工具，分别是get_breakfast_menu，get_lunch_menu，get_dinner_menu。早饭吃什么，午餐吃什么，晚餐吃什么。

In [2]:
from langchain.tools import tool


@tool
def get_breakfast_menu() -> list[str]:
    """Get the breakfast menu."""
    return [
        "包子", "面包", "牛奶", "鸡蛋",
        "馒头",  "米粉"
    ]
@tool
def get_lunch_menu() -> list[str]:
    """Get the lunch menu."""
    return [
        "鱼香肉丝", "红烧肉", "回锅肉", "肝腰合炒",
        "冒菜",  "麻婆豆腐","炝炒凤尾","炝炒丝瓜","小白菜","西红柿炒蛋"
    ]
@tool
def get_dinner_menu() -> list[str]:
    """Get the dinner menu."""
    return [
        "宜宾燃面", "牛肉刀削面", "江油肥肠干饭", "武胜猪肝面",
        "口蘑面",  "kfc"
    ]

创建了一个三餐推荐代理智能体，用于根据用户的提问，调用get_breakfast_menu,get_lunch_menu,get_dinner_menu工具，返回用户的三餐推荐。

In [5]:
from langchain.agents import create_agent

agent = create_agent(
    llm,
    tools=[get_breakfast_menu,get_lunch_menu,get_dinner_menu],
    system_prompt=(
        "You are a helpful meal recommendation agent. Help users decide what to eat "
        "For breakfast, lunch, or dinner based on their preferences. Use the get_breakfast_menu,get_lunch_menu,get_dinner_menu tool "
        "to see available menus."
    ),
)


def reply_text(result) -> str:
    """Return the text of the last message; content can be a string or a list of blocks."""
    content = result["messages"][-1].content
    if isinstance(content, list):
        return "".join(block.get("text", "") for block in content if isinstance(block, dict))
    return content


result = agent.invoke(
    {"messages": [{"role": "user", "content": "What should I eat for breakfast,lunch,dinner today?"}]}
)
# 遍历完整的历史记录，打印每一条消息
print("=== 完整消息历史 ===")
for msg in result["messages"]:
    print(f"\n--- 消息类型: {msg.type} ---")
    print(msg.content)

# 最后的回复
print("\n=== 最终回复 ===")
print(reply_text(result))

=== 完整消息历史 ===

--- 消息类型: human ---
What should I eat for breakfast,lunch,dinner today?

--- 消息类型: ai ---
I'll check all three menus for you today.

--- 消息类型: tool ---
["包子", "面包", "牛奶", "鸡蛋", "馒头", "米粉"]

--- 消息类型: tool ---
["鱼香肉丝", "红烧肉", "回锅肉", "肝腰合炒", "冒菜", "麻婆豆腐", "炝炒凤尾", "炝炒丝瓜", "小白菜", "西红柿炒蛋"]

--- 消息类型: tool ---
["宜宾燃面", "牛肉刀削面", "江油肥肠干饭", "武胜猪肝面", "口蘑面", "kfc"]

--- 消息类型: ai ---
Here are today's menus and my recommendations:

### 🍳 Breakfast
**Options:** 包子 (baozi), 面包 (bread), 牛奶 (milk), 鸡蛋 (egg), 馒头 (mantou), 米粉 (rice noodles)

**Recommendation:** 包子 + 鸡蛋 + 牛奶 — a warm, filling, and protein-balanced combo to start the day.

---

### 🍚 Lunch
**Options:** 鱼香肉丝, 红烧肉, 回锅肉, 肝腰合炒, 冒菜, 麻婆豆腐, 炝炒凤尾, 炝炒丝瓜, 小白菜, 西红柿炒蛋

**Recommendation:** 鱼香肉丝 + 炝炒凤尾 (or 小白菜) — savory and flavorful, balanced with a green vegetable.

---

### 🍜 Dinner
**Options:** 宜宾燃面, 牛肉刀削面, 江油肥肠干饭, 武胜猪肝面, 口蘑面, KFC

**Recommendation:** 牛肉刀削面 — hearty and satisfying without feeling too heavy before bed. (If you want so

当用户输入"hello"这种简单的信息时，智能体不会调用任何工具，会直接回答问题

In [6]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "hello"}]}
)
# 遍历完整的历史记录，打印每一条消息
print("=== 完整消息历史 ===")
for msg in result["messages"]:
    print(f"\n--- 消息类型: {msg.type} ---")
    print(msg.content)

# 最后的回复
print("\n=== 最终回复 ===")
print(reply_text(result))

=== 完整消息历史 ===

--- 消息类型: human ---
hello

--- 消息类型: ai ---
Hello! I'm your meal recommendation assistant. I can help you decide what to eat for breakfast, lunch, or dinner.

What meal are you thinking about? And do you have any preferences (like vegetarian, spicy, light, protein-heavy, etc.)?

=== 最终回复 ===
Hello! I'm your meal recommendation assistant. I can help you decide what to eat for breakfast, lunch, or dinner.

What meal are you thinking about? And do you have any preferences (like vegetarian, spicy, light, protein-heavy, etc.)?


当询问"how is weather today in chengdu"时，由于智能体的工具里面没有任何相关工具，智能体会回答I'd be happy to help, but I don't have access to weather information.没有胡乱回答用户的问题。

In [7]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "how is weather today in chengdu"}]}
)
# 遍历完整的历史记录，打印每一条消息
print("=== 完整消息历史 ===")
for msg in result["messages"]:
    print(f"\n--- 消息类型: {msg.type} ---")
    print(msg.content)

# 最后的回复
print("\n=== 最终回复 ===")
print(reply_text(result))

=== 完整消息历史 ===

--- 消息类型: human ---
how is weather today in chengdu

--- 消息类型: ai ---
I'd be happy to help, but I don't have access to weather information—I can only help with meal recommendations using the breakfast, lunch, and dinner menus.

If you'd like, I can suggest some comforting food ideas for a day like today in Chengdu! 😊 Just let me know if you're looking for breakfast, lunch, or dinner recommendations.

=== 最终回复 ===
I'd be happy to help, but I don't have access to weather information—I can only help with meal recommendations using the breakfast, lunch, and dinner menus.

If you'd like, I can suggest some comforting food ideas for a day like today in Chengdu! 😊 Just let me know if you're looking for breakfast, lunch, or dinner recommendations.


流式输出法

In [4]:
async for token, metadata in agent.astream(
    {"messages": [{"role": "user", "content": "Tell me how to eat a healthy meal"}]},
    stream_mode="messages",
):
    if metadata.get("langgraph_node") == "model" and getattr(token, "content", None):
        print(token.content, end="", flush=True)
print()

I'd be happy to help you plan a healthy meal! To give you specific recommendations, let me check what's currently available across our menus.Based on our current menus, here are my recommendations for a **healthy, balanced meal** for each mealtime.

## 🥗 Breakfast (healthy options)
- **鸡蛋 + 牛奶** → Great protein combo with calcium for bone health
- **馒头 + 牛奶** → A light carb base with protein
- **Tip:** Go for the egg and milk over fried/rich items like 包子 or 米粉 if you want fewer calories.

## 🥦 Lunch (best healthy choices)
- **西红柿炒蛋** → Tomatoes + eggs provide vitamins, lycopene, and lean protein with moderate oil.
- **炝炒丝瓜 / 炝炒凤尾** → Green vegetables, low calorie, high fiber (great for digestion).
- **麻婆豆腐** → Tofu is a good lean protein, though it can be oily/salty—ask for less oil if possible.
- **小白菜** → A light, vitamin-rich green.

**Recommended combo:** 西红柿炒蛋 + 一份炝炒丝瓜或小白菜 (balance protein and vegetables).

## 🍜 Dinner (lighter, cautionary)
Dinner options are heavier/carb- or oil

总结
经过实测，模型在遇到需要具体、实时或受限数据支撑的问题时（比如“今天三餐吃什么”），会主动调用工具去获取答案。当用户只是进行简单的闲聊或常识性对话时（比如“你好”），模型不会调用工具，而是直接利用自身知识回答。而当用户问到工具完全不覆盖的内容（比如问它今天的天气），模型会因为无工具可用而承认自己能力有限。